# 24 · Modern Tabular Pipelines

**2026 learning module** · Run cells from top to bottom. This notebook uses built-in/sample datasets and fixed random seeds where appropriate.

## Goal
Build a leakage-aware mixed-type preprocessing and modeling pipeline using `ColumnTransformer`, `SimpleImputer`, `OneHotEncoder`, and cross-validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.datasets import load_breast_cancer

# A reliable built-in dataset keeps this lesson offline and reproducible.
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()
# Add a categorical feature solely to demonstrate mixed-type preprocessing.
X["size_band"] = pd.cut(X["mean radius"], bins=3, labels=["small", "medium", "large"]).astype("object")
X.loc[X.sample(frac=0.04, random_state=42).index, "mean texture"] = np.nan
X.loc[X.sample(frac=0.03, random_state=7).index, "size_band"] = np.nan
X.head()

In [ ]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
cv = cross_validate(model, X_train, y_train, cv=5, scoring=["accuracy", "f1", "roc_auc"], n_jobs=-1)
{metric: round(float(cv[f"test_{metric}"].mean()), 3) for metric in ["accuracy", "f1", "roc_auc"]}

In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=data.target_names))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=data.target_names, xticks_rotation=30);

## Checklist
- Imputation and scaling are fitted inside the pipeline, preventing train/test leakage.
- `handle_unknown="ignore"` protects inference from unseen categories.
- Cross-validation is performed only on the training split; the test set is held back for final evaluation.